# MNIST FPGA Cloud Classification Measurements

This notebook documents the **FPGA-to-Cloud Run measurement workflow** used for MNIST classification.

The experiments were performed in two stages:

1. **Formal cloud accuracy, latency, and throughput measurement**
   - controlled MNIST test images;
   - 100-image batches;
   - persistent TCP/TLS connection;
   - per-image latency records;
   - accuracy and error counts;
   - whole-run timing.

2. **Cloud power and memory measurement**
   - separate FPGA build;
   - local neural-network hardware removed;
   - controlled MNIST workload kept;
   - cloud communication path kept;
   - power and memory measured with unnecessary local inference hardware excluded.

This separation prevents the power/memory measurements from being inflated by hardware that was not required for cloud inference.


## 1. System Architecture

The cloud-assisted path used the ZedBoard programmable logic (PL) and processing system (PS) together.

```text
MNIST test image in PL
        ↓
shared AXI BRAM mailbox
        ↓
Zynq PS
        ↓
lwIP / TCP
        ↓
Mbed TLS
        ↓
HTTPS POST /predict
        ↓
Google Cloud Run
        ↓
MNIST neural network
        ↓
JSON prediction
        ↓
PS writes result to mailbox
        ↓
PL consumes result
```

The Cloud Run service expected exactly 784 bytes per image and returned a JSON prediction.


## 2. SDK / Network Setup

The cloud measurement build required additional work in Xilinx SDK because the original FPGA project did not yet include the complete network/TLS application path.

The SDK application used a standalone Zynq ARM project named:

```text
network_tls_test
```

The application was created against the exported Vivado hardware platform and an lwIP-enabled BSP.

The application project used an **Empty Application** starting point, then added the platform/network source needed for the ZedBoard application, including:

```text
platform.c
platform.h
platform_config.h
main.c
tcp_client.c
```

The SDK project directory was reused during later development, so the formal measurement versions of `main.c` and `tcp_client.c` should be used for reproducing the controlled 100-image experiment.


## 3. lwIP Networking Requirements

The PS application used Xilinx lwIP for Ethernet, DHCP, DNS, and TCP.

The application startup sequence was:

```text
initialize platform
        ↓
initialize lwIP
        ↓
add ZedBoard Ethernet interface
        ↓
enable interrupts
        ↓
start DHCP
        ↓
obtain IP / netmask / gateway
        ↓
initialize DNS
        ↓
resolve Cloud Run hostname
        ↓
open TCP connection
```

The network loop continuously serviced:

```text
xemacif_input()
tcp_fasttmr()
tcp_slowtmr()
dns_tmr()
transfer_data()
```

This polling was required for DHCP, DNS, TCP, TLS, and the mailbox-driven cloud state machine to continue progressing.


## 4. DNS Configuration and Application-Level `dns.c`

DNS required additional configuration beyond the initial lwIP setup.

For the working project:

- DNS support was enabled with `LWIP_DNS=1`;
- a copy of lwIP `dns.c` was compiled directly into the application project;
- `dns_init()` was called explicitly;
- `dns_tmr()` was driven from the main polling loop; and
- the DNS server was manually set to:

```text
1.1.1.1
```

The Cloud Run hostname was then resolved with `dns_gethostbyname()`.

This application-level `dns.c` workaround was part of the working network configuration and should be included when reconstructing the SDK project.


## 5. Mbed TLS Integration

HTTPS communication required adding Mbed TLS to the SDK application.

The project used **Mbed TLS 2.28** headers and library support.

The configuration used during the completed MNIST experiments disabled components that were not needed or were incompatible with the bare-metal environment:

```text
MBEDTLS_NET_C
MBEDTLS_TIMING_C
MBEDTLS_FS_IO
MBEDTLS_CERTS_C
```

The TLS application itself used:

```text
mbedtls_ssl
mbedtls_entropy
mbedtls_ctr_drbg
mbedtls_error
mbedtls_memory_buffer_alloc
```

The network transport was provided by lwIP TCP rather than Mbed TLS socket code.


## 6. Development TLS Configuration

The completed MNIST measurement client used encryption, but the TLS configuration was still a development configuration.

The client used:

```text
MBEDTLS_SSL_VERIFY_NONE
```

and a deterministic development entropy source.

This means:

- TLS encryption was active;
- the Cloud Run hostname was supplied through SNI;
- server-certificate verification was disabled; and
- the entropy source was not suitable for a production security implementation.

These choices should be described as part of the experimental implementation rather than as production security recommendations.


## 7. SDK Mailbox Interface

The SDK application communicated with the programmable logic through the AXI BRAM mailbox beginning at:

```text
0x40000000
```

For each image, the PS:

```text
waits for STATUS = IMAGE_READY
        ↓
reads 196 32-bit words
        ↓
unpacks 784 image bytes
        ↓
reads image index and expected label
        ↓
sends HTTPS request
        ↓
receives prediction
        ↓
writes RESULT
        ↓
writes STATUS = RESULT_READY
```

The formal measurement client also stored summary counters, latency information, and per-image records back into the mailbox.


## 8. Persistent TCP/TLS Session

TCP and TLS setup occurred before the controlled workload began.

The sequence was:

```text
DHCP
  ↓
DNS
  ↓
TCP connection
  ↓
TLS handshake
  ↓
100-image experiment
```

After the TLS handshake succeeded, the same connection was reused for the image requests.

Each HTTP request used:

```text
Connection: keep-alive
```

and response completion was detected from `Content-Length` rather than by waiting for the server to close the TCP connection.

This was necessary so the formal measurements represented repeated cloud inference over a persistent connection instead of including a new TCP/TLS setup for every image.


## 9. Preparing Each 100-Image FPGA Run

Each controlled 100-image block required a matching FPGA test-image ROM and expected-label package.

For each 100-image block:

1. generate the BRAM initialization data for the selected MNIST images;
2. generate the matching expected-label VHDL package;
3. update the Block Memory Generator contents;
4. rebuild the Vivado design;
5. generate the corresponding bitstream;
6. export/update the hardware platform used by SDK when the Vivado hardware changed;
7. build the `network_tls_test` SDK application against that hardware platform; and
8. program/run the FPGA and PS application.

The hardware logic, mailbox protocol, Cloud Run service, model, and SDK measurement logic otherwise remained unchanged across the controlled blocks.


## 10. Running One Formal Measurement

A formal run followed this sequence:

```text
program FPGA
        ↓
launch SDK application / ELF
        ↓
wait for DHCP
        ↓
wait for DNS resolution
        ↓
wait for TCP connection
        ↓
wait for TLS handshake
        ↓
start the 100-image workload
        ↓
allow all 100 requests to complete
        ↓
check TEST_COMPLETE
        ↓
collect results
```

The connection-establishment phase occurred before the measured 100-image workload.

For the formal dataset, one successful validation execution was performed first, followed by repeated formal runs for each 100-image block.

A valid run required:

```text
100 completed images
aligned per-image records
zero communication errors
valid TEST_COMPLETE
```

Pre-workload connection failures or incomplete/misaligned runs were not treated as valid measurement runs.


## 11. Result-Dump Behavior

During development, the SDK application could print the stored BRAM measurement data over UART after the run completed.

The verification version used:

```text
AUTO_DUMP_RESULTS = 1
```

The final quiet measurement build used:

```text
AUTO_DUMP_RESULTS = 0
```

The result dump was intentionally placed after `TEST_COMPLETE`, so UART printing did not participate in the measured 100-image interval.

The stored BRAM records remained the primary experiment data.


## 12. Measurement Project Order

The measurement work was performed in this order:

```text
Cloud speed / throughput / accuracy build
        ↓
100-image controlled test batches
        ↓
formal accuracy + latency + throughput results
        ↓
separate cloud power build
        ↓
remove local neural-network hardware
        ↓
measure cloud-assisted power and memory
```

The two FPGA configurations therefore serve different purposes and should not be mixed when reporting results.


## 13. Formal Cloud Measurement Build

The first cloud measurement configuration kept the full measurement instrumentation required to evaluate:

- prediction accuracy;
- per-image cloud latency;
- minimum latency;
- maximum latency;
- latency sum;
- whole 100-image run time;
- number of successful classifications;
- number of incorrect classifications; and
- communication errors.

The PS application used a shared mailbox at:

```text
0x40000000
```

and stored summary and per-image results back into BRAM.


## 14. Controlled 100-Image Workload

The formal cloud experiment used controlled MNIST images rather than live camera input.

Each image contained:

```text
784 pixels
1 expected label
```

The FPGA published one image at a time through the shared mailbox. The PS read the 784 bytes, sent them to the Cloud Run endpoint, parsed the returned prediction, and wrote the result back to the PL.

The formal workload size was:

```text
100 images per test run
```

Multiple 100-image blocks were used to cover the first 1,000 MNIST test images.


## 15. Mailbox Measurement Map

The cloud measurement client used the following mailbox fields:

```text
0x000 - 0x30C   packed 784-byte image
0x310           STATUS
0x314           image index
0x318           prediction result
0x31C           expected label

0x320           total count
0x324           correct count
0x328           error count

0x32C           minimum latency (µs)
0x330           maximum latency (µs)

0x334           latency sum, low 32 bits
0x338           latency sum, high 32 bits

0x33C           total 100-image run time (µs)
0x340           TEST_COMPLETE
```

Per-image measurement records began at:

```text
0x400 + image_index × 0x10
```

Each record stored:

```text
image index
expected / predicted / correct / error flags
latency in microseconds
raw latency timer ticks
```


## 16. Network and TLS Path

The PS application performed:

```text
DHCP
  ↓
DNS lookup
  ↓
TCP connection to Cloud Run
  ↓
Mbed TLS initialization
  ↓
TLS handshake
  ↓
persistent HTTPS connection
```

The Cloud Run host used during the MNIST experiments was:

```text
mnist-cloud-inference-221036165238.us-east1.run.app
```

The connection used:

```text
TCP port 443
HTTPS
Connection: keep-alive
```

The persistent connection was important because the formal experiment measured repeated inference requests without opening a new TCP/TLS connection for every image.


## 17. HTTP Request

For each image, the PS constructed an HTTP request with:

```text
POST /predict HTTP/1.1
Content-Type: application/octet-stream
Content-Length: 784
Connection: keep-alive
```

The body contained the 784 MNIST input bytes.

The PS then waited for a complete HTTP response and parsed:

```json
{"prediction": N}
```

where `N` was the predicted MNIST digit.


## 18. Per-Image Latency Measurement

Per-image latency was measured on the Zynq PS using `XTime`.

Timing started immediately before the first TLS write belonging to the request.

```text
request_start
    ↓
send HTTP header
    ↓
send 784-byte image
    ↓
Cloud Run inference
    ↓
receive complete HTTP response
    ↓
request_end
```

Latency therefore represents **client-side cloud request latency**, including:

- request transmission;
- network delay;
- Cloud Run processing;
- inference;
- response transmission; and
- response receipt.

It does not represent Cloud Run server-only execution time.


## 19. Whole-Run Timing and Throughput

A second timer measured the complete 100-image run.

The whole-run timer started immediately before the first HTTPS request for image 0 and ended after the final prediction for image 99 had been written back and acknowledged through the mailbox.

This measurement included:

- all 100 HTTPS requests;
- all 100 responses;
- inter-image control overhead; and
- the persistent connection state.

Effective throughput was calculated from the total run time:

```text
throughput = 100 / total_run_time_seconds
```


In [ ]:
# Example throughput calculation.
# Replace total_run_us with the measured 100-image total run time.

total_run_us = None

if total_run_us:
    total_run_s = total_run_us / 1_000_000.0
    throughput = 100.0 / total_run_s
    print(f"Throughput: {throughput:.3f} classifications/s")
else:
    print("Enter the measured total_run_us value.")


## 20. Accuracy Measurement

For each returned prediction, the PS compared:

```text
predicted digit
vs.
expected MNIST label
```

The measurement client tracked:

```text
total_count
correct_count
error_count
```

A classification mismatch counted as an incorrect prediction.

Communication or response failures were counted separately as errors.

The first 1,000 MNIST test images were evaluated in controlled 100-image blocks.


## 21. Formal Cloud Accuracy Result

The final cloud classification result across the first 1,000 MNIST test images was:

```text
970 / 1000 correct
97.0% accuracy
```

This result should be reported separately from the local FPGA classification accuracy because the cloud service executes the floating-point neural-network model rather than the fixed-point ZyNet hardware implementation.


## 22. Formal Cloud Latency and Throughput Results

The formal cloud runs produced client-side latency values in the approximate steady-state range of:

```text
58-60 ms per request
```

The complete formal dataset included a wider observed range, with approximately:

```text
minimum: 51.934 ms
maximum: 159.652 ms
```

The resulting sustained throughput was approximately:

```text
16.8-17.1 classifications/s
```

The first request after startup or a cold service state should be treated separately from steady-state requests.


## 23. Why Power and Memory Used a Separate FPGA Build

After the formal cloud accuracy/latency/throughput measurements were complete, a separate FPGA configuration was created for power and memory measurements.

The purpose was to remove hardware that was **not required for cloud classification**.

The cloud inference path requires:

```text
controlled image source
shared mailbox
Zynq PS
Ethernet
TCP
TLS
HTTPS
Cloud Run
```

It does **not** require the local ZyNet inference engine.

Therefore, the local neural-network hardware was removed before the cloud power/memory measurements.


## 24. Cloud Power Build

The cloud power build kept:

- controlled MNIST test-image ROM;
- image-selection/control logic;
- shared AXI BRAM mailbox;
- Zynq Processing System;
- Ethernet/network stack;
- TCP;
- Mbed TLS;
- HTTPS request handling; and
- Cloud Run communication.

The local neural-network inference hardware was removed.

This makes the power measurement more representative of the actual FPGA-assisted cloud workload rather than a combined local-plus-cloud design.


## 25. Why the Test ROM Remained

The controlled MNIST ROM was intentionally kept in the power configuration.

It provided a repeatable source of input data without requiring:

- live camera interaction;
- manual image capture;
- local classification hardware; or
- user timing.

The ROM therefore served as a deterministic workload generator for repeated cloud requests.


## 26. Memory Measurement

The cloud-assisted configuration requires significant PS-side memory for networking and TLS.

Important software buffers included:

```text
TLS receive buffer        32 KB
Mbed TLS memory pool     128 KB
HTTP response buffer       4 KB
HTTP header buffer       ~512 B
784-byte image buffer
```

Additional memory was consumed by:

- lwIP;
- TCP state;
- DNS;
- DHCP;
- Mbed TLS structures;
- stack;
- heap; and
- application state.

Memory results from this configuration should therefore be interpreted as the memory cost of the **cloud communication stack plus application logic**, rather than the neural-network model itself.


## 27. Power Measurement Boundary

The cloud power experiment measures the board while it performs the cloud-assisted workload.

The measured power includes:

- Zynq PS activity;
- FPGA control logic;
- AXI/mailbox activity;
- Ethernet;
- software networking;
- TLS processing; and
- repeated cloud requests.

The remote Cloud Run server's electrical power is **not** included in the board-level physical power measurement.

The cloud service is therefore outside the local power boundary.


## 28. Source Configurations

The Vivado-side source files associated with the two cloud measurement configurations are:

```text
top_cloud_speed.vhd
mnist_test_rom_pkg_cloud_speed.vhd
blk_mem_gen_mnist_test_cloud_speed.xcix
design_1_bd_cloud_speed.tcl
[cloud speed bitstream]

top_cloud_power.vhd
mnist_test_rom_pkg_cloud_power.vhd
blk_mem_gen_mnist_test_cloud_power.xcix
design_1_bd_cloud_power.tcl
[cloud power bitstream]
```

The formal SDK measurement application is represented by the 100-image cloud measurement versions of:

```text
main.c
tcp_client.c
```

The SDK project directory was reused later for live camera-to-cloud work, so later copies of `tcp_client.c` should not be substituted for the formal 100-image measurement version.


## 29. Separation from Live Cloud Classification

This notebook documents the **formal cloud measurement experiments**.

It does not document the later live camera-to-cloud implementation.

The live branch uses:

```text
frozen camera image
    ↓
mailbox
    ↓
PS
    ↓
Cloud Run
    ↓
prediction returned to PL
```

and treats latency as a functional diagnostic rather than the formal 100-image measurement dataset.

That implementation belongs in the separate live cloud branches.


## 30. Recommended Supporting Data

The corresponding Notebook-and-Data branch should include, where available:

```text
formal UART measurement output
100-image result dumps
CSV or text summaries
Cloud Run request logs
power photographs/videos
memory/ELF size output
Vivado utilization reports
Vivado power reports
screenshots of measurement runs
```

The code branch should contain the source and bitstreams, while the Notebook-and-Data branch should contain the evidence and measured results.


## 31. Experimental Summary

The cloud measurement workflow was deliberately split into two phases.

### Phase 1 — Accuracy, Latency, Throughput

```text
controlled MNIST images
        ↓
100-image batches
        ↓
persistent HTTPS connection
        ↓
Cloud Run inference
        ↓
per-image latency + accuracy records
        ↓
whole-run throughput
```

### Phase 2 — Power and Memory

```text
controlled MNIST images
        ↓
local NN hardware removed
        ↓
same cloud communication path
        ↓
repeatable cloud workload
        ↓
power + memory measurement
```

This separation allows the formal cloud performance results to be reported independently from the hardware and software resource cost of the cloud-assisted configuration.
